# Merge and re-split an entity-matching dataset

In [ ]:
# ===================== ALL PARAMETERS - CHANGE THEM ONLY HERE =====================
DATASET      = "WalmartAmazon"                 # folder name under Data_preparation/Data/
PROJECT_ROOT = "/home/u59352/SolvIER2"
                      # e.g. "/home/u59352/SolvIER-main"; None -> auto-detect
SOURCE_SPLIT = "test"                    # the 90% file every later step is built from

# Step 1 - merge the _original files and re-split them
SEED   = 584                 #584, 1234, 1576, 2534, 7430
RATIOS = {"test": 0.60, "train": 0.20, "valid": 0.20}  # largest split absorbs the rounding remainder

# Step 3 - null injection
MECHANISMS   = ["MCAR", "MAR", "MNAR"]
PCTS         = [0.05, 0.05, 0.1, 0.1, 0.1]    # INCREMENTS as fractions -> cumulative levels 5/10/20/40%
NULL_SEED    = SEED
STRENGTH     = 1.0                       # slope on the standardised driver; 0 collapses MAR/MNAR to MCAR
MAR_OBSERVED = None                      # MAR: columns kept fully observed. None -> the numeric ones
DROP_STALE   = True                      # delete derived files of levels no longer listed above

# Step 4 - imputation. IMPUTER goes into the file names, so the imputed files
# sit beside the nonimputed ones: ..._nonimputed.csv / ..._imputed_senti.csv
IMPUTER = "senti"                       # "senti" | "knn" | "simple"

# senti only (ignored by knn and simple)
EMBED_MODEL = "sentence-transformers/LaBSE"
MEDIAN_K    = 25                        # neighbours retrieved per row
TOP_K       = 10                         # of those, how many with a value in the target column vote
SIM_TH      = 0.94                       # minimum cosine similarity to vote
HIGH_TH     = 1.0                        # at or above this, copy the neighbour's value directly

# knn only (ignored by senti and simple)
KNN_K   = 10                             # neighbours that vote per cell
# ==================================================================================

import os
import random
import sys
from collections import Counter
from pathlib import Path

# Jupyter does not always start in this notebook's folder (on a cluster the kernel
# often starts in $HOME), so locate the project instead of trusting the cwd.
_hint = PROJECT_ROOT or os.environ.get("SOLVIER_ROOT")
_cands = [Path(_hint).expanduser().resolve()] if _hint else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
_roots = [p for p in _cands if (p / "Data_preparation" / "Data").is_dir()]
assert _roots, (f"project root not found from {Path.cwd()}. Set PROJECT_ROOT above to the "
                f"SolvIER2 folder, e.g. PROJECT_ROOT = \"/home/u59352/SolvIER2\"")
CODE_DIR = _roots[0] / "Data_preparation" / "code"
assert (CODE_DIR / "context.py").is_file(), f"context.py is missing - upload it to {CODE_DIR}"
sys.path.insert(0, str(CODE_DIR))
from context import ID_COLS, SIDES, SPLITS, levels_of, steps_of, write_params  # noqa: E402
from imputers import STRATEGIES  # noqa: E402

assert IMPUTER in STRATEGIES, f"IMPUTER must be one of {STRATEGIES}, got {IMPUTER!r}"

ROOT = CODE_DIR.parents[1]
DATA_DIR = ROOT / "Data_preparation" / "Data" / DATASET
if not DATA_DIR.is_dir():
    available = sorted(d.name for d in (ROOT / "Data_preparation" / "Data").iterdir() if d.is_dir())
    raise FileNotFoundError(f"{DATA_DIR} not found. Datasets available: {available or 'none'}")

PCT_LEVELS = levels_of(PCTS)  # cumulative levels, as used in every file name
STEPS = steps_of(PCTS)        # level -> the increment that produced it
PARAMS_FP = write_params(ROOT, {"dataset": DATASET, "source_split": SOURCE_SPLIT, "seed": SEED,
                                "ratios": RATIOS, "mechanisms": MECHANISMS, "increments": PCTS,
                                "null_seed": NULL_SEED, "strength": STRENGTH,
                                "mar_observed": MAR_OBSERVED, "drop_stale": DROP_STALE,
                                "imputer": IMPUTER, "knn_k": KNN_K,
                                "embed_model": EMBED_MODEL, "median_k": MEDIAN_K, "top_k": TOP_K,
                                "sim_th": SIM_TH, "high_th": HIGH_TH})
print("project root:", ROOT)
print("data dir    :", DATA_DIR)
print("null levels :", PCT_LEVELS, "% cumulative, from increments", PCTS)
print("imputer     :", IMPUTER)
print("parameters  ->", PARAMS_FP.name)

# merge the three _original files, tagging each row with the split it came from
per_split = {s: [l.rstrip("\n") for l in (DATA_DIR / f"{s}_{DATASET}_original.txt").open(encoding="utf-8") if l.strip()]
             for s in SPLITS}
merged = [f"{line}\t{s}" for s in SPLITS for line in per_split[s]]
(DATA_DIR / f"{DATASET}_merged.txt").write_text("\n".join(merged) + "\n", encoding="utf-8")

# shuffle, spreading labels evenly so the tiny train/valid splits still get positives
rows = [l.rsplit("\t", 1)[0] for l in merged]
rng = random.Random(SEED)
rng.shuffle(rows)
groups: dict[str, list[str]] = {}
for r in rows:
    groups.setdefault(r.rsplit("\t", 1)[1], []).append(r)
rows = [r for _, r in sorted((i / len(g), r) for g in groups.values() for i, r in enumerate(g))]

# cut 90 / 5 / 5 and write the new splits
sizes = {s: int(len(rows) * RATIOS[s]) for s in RATIOS}
sizes[max(RATIOS, key=RATIOS.get)] += len(rows) - sum(sizes.values())
start = 0
for s in RATIOS:
    part = rows[start : start + sizes[s]]
    start += sizes[s]
    rng.shuffle(part)
    (DATA_DIR / f"{s}_{DATASET}.txt").write_text("\n".join(part) + "\n", encoding="utf-8")
    pos = sum(1 for r in part if r.endswith("\t1"))
    print(f"{s:>5}: {len(part):>4} rows ({len(part) / len(rows):5.1%})  positives: {pos} ({pos / len(part):.1%})")

assert start == len(rows) and Counter(rows) == Counter(l.rsplit("\t", 1)[0] for l in merged)


## Step 2 - the 60% test file to CSV, then to one row per entity

In [ ]:
# --- every parameter lives in the first cell; recovered from disk if this cell runs alone ---
if "PCT_LEVELS" not in globals():
    import os, sys
    from pathlib import Path
    _hint = os.environ.get("SOLVIER_ROOT")
    _cands = [Path(_hint).expanduser().resolve()] if _hint else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    _roots = [p for p in _cands if (p / "Data_preparation" / "Data").is_dir()]
    assert _roots, "run the first cell (it has PROJECT_ROOT), or export SOLVIER_ROOT=/path/to/SolvIER-main"
    _code = _roots[0] / "Data_preparation" / "code"
    assert (_code / "context.py").is_file(), f"context.py is missing - upload it to {_code}"
    sys.path.insert(0, str(_code))
    from context import load_context
    globals().update(load_context())
    print("settings recovered from", PARAMS_FP.name)

import re

import pandas as pd

SRC = DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}.txt"


def parse(entity: str) -> dict[str, str]:
    """'COL title VAL x COL year VAL 2002' -> {'title': 'x', 'year': '2002'}"""
    fields = (f.partition(" VAL ") for f in re.split(r"\bCOL\b", entity) if f.strip())
    return {k.strip(): v.strip() for k, _, v in fields}


pairs = []
for pair_id, line in enumerate((l for l in SRC.open(encoding="utf-8") if l.strip()), start=1):
    left, right, label = line.rstrip("\n").split("\t")
    pairs.append({"pair_id": pair_id, "left": parse(left), "right": parse(right), "label": int(label)})

ATTRS = list(pairs[0]["left"])  # attribute order, needed to rebuild the COL/VAL strings

# one row per pair
wide = pd.DataFrame([
    {"split": SOURCE_SPLIT, "pair_id": p["pair_id"],
     **{f"{side}_{a}": p[side].get(a, "") for side in SIDES for a in ATTRS},
     "label": p["label"]}
    for p in pairs
])

# cut into the left and right halves, stack them into one row per entity
long = pd.concat([
    wide[[*ID_COLS, *[f"{side}_{a}" for a in ATTRS]]]
    .rename(columns={f"{side}_{a}": a for a in ATTRS})
    .assign(side=side)
    for side in SIDES
], ignore_index=True)[["split", "pair_id", "side", *ATTRS, "label"]]
long = long.sort_values(["pair_id", "side"], ascending=[True, True]).reset_index(drop=True)

for name, df in (("wide", wide), ("long", long)):
    out = DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}_{name}.csv"
    df.to_csv(out, index=False)
    print(f"{name:>4}: {len(df):>4} rows x {len(df.columns)} cols -> {out.name}")

# nulls already present in the source: Ditto `COL x VAL ` with nothing after the
# VAL parses as the empty string, so an attribute can be effectively null in the
# ground truth itself. Report both: `empty` is the count that score_logs later
# treats as "no_truth"; NaN appears when a row is missing the attribute entirely.
blank = long[ATTRS].isna() | long[ATTRS].astype("string").fillna("").eq("")
total = len(long)
print(f"\npre-existing nulls in {SOURCE_SPLIT}_{DATASET}_long.csv  ({total} entity rows)")
print(f"  {'column':<16}  {'empty':>8}  {'NaN':>6}  {'empty %':>8}")
for a in ATTRS:
    n_empty = int(blank[a].sum())
    n_nan = int(long[a].isna().sum())
    print(f"  {a:<16}  {n_empty:>8}  {n_nan:>6}  {100 * n_empty / total:>7.1f}%")

# the entity rows must fold back into the exact source lines
rebuilt = [
    "\t".join(" ".join(f"COL {a} VAL {row[a]}" for a in ATTRS) for _, row in grp.iterrows())
    + f"\t{grp['label'].iloc[0]}"
    for _, grp in long.groupby("pair_id", sort=True)
]
original = [" ".join(l.split()) for l in SRC.read_text(encoding="utf-8").splitlines() if l.strip()]
assert [" ".join(r.split()) for r in rebuilt] == original, "entity rows do not fold back into the source"
print(f"\nall {len(rebuilt)} pairs fold back into {SRC.name}")
long.head()

---
## Step 3 - null injection (MCAR / MAR / MNAR)


In [ ]:
# --- every parameter lives in the first cell; recovered from disk if this cell runs alone ---
if "PCT_LEVELS" not in globals():
    import os, sys
    from pathlib import Path
    _hint = os.environ.get("SOLVIER_ROOT")
    _cands = [Path(_hint).expanduser().resolve()] if _hint else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    _roots = [p for p in _cands if (p / "Data_preparation" / "Data").is_dir()]
    assert _roots, "run the first cell (it has PROJECT_ROOT), or export SOLVIER_ROOT=/path/to/SolvIER-main"
    _code = _roots[0] / "Data_preparation" / "code"
    assert (_code / "context.py").is_file(), f"context.py is missing - upload it to {_code}"
    sys.path.insert(0, str(_code))
    from context import load_context
    globals().update(load_context())
    print("settings recovered from", PARAMS_FP.name)

import numpy as np
import pandas as pd

if "long" not in globals():  # Step 2 not run in this kernel - read its output instead
    long = pd.read_csv(DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}_long.csv", dtype=str)


def proxy(s: pd.Series) -> np.ndarray:
    """Standardised numeric view of a column: its value if numeric, else string length."""
    num = pd.to_numeric(s, errors="coerce")
    x = num.astype(float) if num.notna().sum() >= 0.9 * s.notna().sum() else s.astype("string").str.len().astype(float)
    return np.clip(((x - x.mean()) / (x.std(ddof=0) or 1.0)).fillna(0.0).to_numpy(), -3, 3)


def plan(df: pd.DataFrame, mech: str) -> tuple[list[str], dict[str, list[str]]]:
    """Which columns get masked, and which columns drive each one's missingness."""
    attrs = [c for c in df.columns if c not in ID_COLS + ["side"]]
    if mech == "MCAR":
        return attrs, {c: [] for c in attrs}
    if mech == "MNAR":
        return attrs, {c: [c] for c in attrs}
    observed = MAR_OBSERVED or [c for c in attrs if pd.to_numeric(df[c], errors="coerce").notna().all()] or attrs[-1:]
    targets = [c for c in attrs if c not in observed]
    assert targets, f"MAR needs a column left to mask, observed={observed}"
    return targets, {c: list(observed) for c in targets}


def inject(df: pd.DataFrame, mech: str, levels: list[float], seed: int) -> dict[float, pd.DataFrame]:
    """One fixed per-column ordering, masked deeper at each level, so the levels nest.

    `levels` are cumulative percentages. Level i masks the same cells as level i-1
    plus the next slice of the same ordering - never a fresh draw.
    """
    attrs = [c for c in df.columns if c not in ID_COLS + ["side"]]
    targets, drivers = plan(df, mech)
    rng = np.random.default_rng(seed)
    w = {c: np.exp(STRENGTH * np.mean([proxy(df[d]) for d in drivers[c]], axis=0)) if drivers[c] else 1.0
         for c in targets}
    keys = {c: rng.random(len(df)) ** (1.0 / w[c]) for c in targets}
    order = {c: np.argsort(-keys[c], kind="stable") for c in targets}  # highest key masked first

    out = {}
    for pct in levels:
        k = round(pct / 100 * len(df) * len(attrs) / len(targets))  # budget spread over the target columns
        assert k <= len(df), f"{mech} {pct}% needs {k} of {len(df)} rows in each of {len(targets)} columns"
        masked = df.copy()
        for c in targets:
            masked.loc[df.index[order[c][:k]], c] = np.nan
        out[pct] = masked
    return out


ATTR_COLS = [c for c in long.columns if c not in ID_COLS + ["side"]]
print(f"increments {PCTS} -> cumulative levels {PCT_LEVELS} (%)\n")

if DROP_STALE:  # files from a previous run whose level is no longer requested
    keep = {str(p) for p in PCT_LEVELS}
    for fp in sorted(DATA_DIR.glob(f"{SOURCE_SPLIT}_{DATASET}_long_*")):
        parts = fp.stem.split("_")
        if str(NULL_SEED) in parts and parts[parts.index(str(NULL_SEED)) - 1] not in keep:
            print("removed stale", fp.name)
            fp.unlink()

summary = []
for mech in MECHANISMS:
    levels = inject(long, mech, PCT_LEVELS, NULL_SEED)
    for pct, df in levels.items():
        out = DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}_long_{mech}_{pct}_{NULL_SEED}_nonimputed.csv"
        df.to_csv(out, index=False)
        summary.append({"mechanism": mech, "added %": STEPS[pct], "cumulative %": pct,
                        "actual %": round(df[ATTR_COLS].isna().to_numpy().mean() * 100, 1),
                        "nulls": int(df[ATTR_COLS].isna().to_numpy().sum()), "file": out.name})
    # each level must contain the one below it, and the id columns must survive untouched
    for lo, hi in zip(PCT_LEVELS, PCT_LEVELS[1:]):
        assert (levels[lo].isna() <= levels[hi].isna()).all().all(), f"{mech}: {lo}% is not a subset of {hi}%"
        added = int((levels[hi][ATTR_COLS].isna() & ~levels[lo][ATTR_COLS].isna()).to_numpy().sum())
        assert added > 0, f"{mech}: {lo}% -> {hi}% added no nulls"
    assert levels[max(PCT_LEVELS)][ID_COLS + ["side"]].equals(long[ID_COLS + ["side"]]), f"{mech} touched the id columns"
    if mech == "MAR":
        observed = [c for c in ATTR_COLS if c not in plan(long, "MAR")[0]]
        assert levels[max(PCT_LEVELS)][observed].notna().all().all(), "MAR observed columns must stay fully observed"

pd.DataFrame(summary)

---
## Step 4 - imputation


In [ ]:
# --- every parameter lives in the first cell; recovered from disk if this cell runs alone ---
if "PCT_LEVELS" not in globals():
    import os, sys
    from pathlib import Path
    _hint = os.environ.get("SOLVIER_ROOT")
    _cands = [Path(_hint).expanduser().resolve()] if _hint else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    _roots = [p for p in _cands if (p / "Data_preparation" / "Data").is_dir()]
    assert _roots, "run the first cell (it has PROJECT_ROOT), or export SOLVIER_ROOT=/path/to/SolvIER-main"
    _code = _roots[0] / "Data_preparation" / "code"
    assert (_code / "context.py").is_file(), f"context.py is missing - upload it to {_code}"
    sys.path.insert(0, str(_code))
    from context import load_context
    globals().update(load_context())
    print("settings recovered from", PARAMS_FP.name)

import sys

sys.path.insert(0, str(ROOT / "Data_preparation" / "code"))
from imputers import run_imputation  # noqa: E402

IMP_PREFIX = f"{SOURCE_SPLIT}_{DATASET}_long"   # the {prefix}_{MECH}_{pct}_{seed}_nonimputed.csv files

imp_summary = run_imputation(
    data_dir=DATA_DIR,
    dataset=IMP_PREFIX,
    mechanisms=MECHANISMS,
    pcts=PCT_LEVELS,
    seed=NULL_SEED,
    id_cols=ID_COLS + ["side"],                 # never used as features, never imputed
    strategy=IMPUTER,
    ground_truth_file=DATA_DIR / f"{IMP_PREFIX}.csv",   # scoring only
    senti_code=ROOT / "SENTI" / "Code",
    embed_model=EMBED_MODEL,
    median_k=MEDIAN_K,
    top_k=TOP_K,
    sim_th=SIM_TH,
    high_th=HIGH_TH,
    knn_k=KNN_K,                                        # knn only; ignored otherwise
)
imp_summary

---
## Step 5 - fold the long CSVs back to Ditto format


In [ ]:
# --- every parameter lives in the first cell; recovered from disk if this cell runs alone ---
if "PCT_LEVELS" not in globals():
    import os, sys
    from pathlib import Path
    _hint = os.environ.get("SOLVIER_ROOT")
    _cands = [Path(_hint).expanduser().resolve()] if _hint else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    _roots = [p for p in _cands if (p / "Data_preparation" / "Data").is_dir()]
    assert _roots, "run the first cell (it has PROJECT_ROOT), or export SOLVIER_ROOT=/path/to/SolvIER-main"
    _code = _roots[0] / "Data_preparation" / "code"
    assert (_code / "context.py").is_file(), f"context.py is missing - upload it to {_code}"
    sys.path.insert(0, str(_code))
    from context import load_context
    globals().update(load_context())
    print("settings recovered from", PARAMS_FP.name)

import shutil
import sys

import pandas as pd

sys.path.insert(0, str(ROOT / "Data_preparation" / "code"))
from imputers import STRATEGIES  # noqa: E402

VARIANTS = ["nonimputed"] + [f"imputed_{s}" for s in STRATEGIES]  # whichever have files

DITTO_DIR = DATA_DIR / f"{DATASET}_ditto"
SOLVER_DATA_DIR = ROOT / "SolvIER" / "Data"  # what train_ditto.py / matcher.py read
DITTO_DIR.mkdir(exist_ok=True)
assert SOLVER_DATA_DIR.is_dir(), f"SolvER data folder not found at {SOLVER_DATA_DIR}"


def to_ditto(csv_fp: Path) -> Path:
    """Fold one long CSV back into Ditto's pair-per-line .txt. Returns the path."""
    df = pd.read_csv(csv_fp, dtype=str).fillna("")
    lines = []
    for pair_id, grp in df.groupby("pair_id", sort=False):
        grp = grp.set_index("side")
        assert set(grp.index) == set(SIDES), f"{csv_fp.name}: pair {pair_id} has sides {list(grp.index)}"
        entities = [" ".join(f"COL {a} VAL {grp.at[side, a]}" for a in ATTRS) for side in SIDES]
        labels = set(grp["label"])
        assert len(labels) == 1, f"{csv_fp.name}: pair {pair_id} has labels {labels}"
        lines.append("\t".join(entities) + f"\t{labels.pop()}")
    out = DITTO_DIR / (csv_fp.stem.replace("_long", "") + ".txt")
    out.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return out


# the untouched long CSV must fold back to the exact source file - checks the converter
check = to_ditto(DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}_long.csv")
norm = lambda p: [" ".join(l.split()) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
assert norm(check) == norm(DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}.txt"), "converter does not reproduce the source"
print(f"converter check: {check.name} matches {SOURCE_SPLIT}_{DATASET}.txt\n")
check.unlink()  # the clean test file is copied in below from the split itself

if DROP_STALE:  # .txt files of levels that are no longer requested
    keep = {str(p) for p in PCT_LEVELS}
    for d in (DITTO_DIR, SOLVER_DATA_DIR):
        for fp in sorted(d.glob(f"*_{DATASET}_*_{NULL_SEED}_*.txt")):
            parts = fp.stem.split("_")
            if str(NULL_SEED) in parts and parts[parts.index(str(NULL_SEED)) - 1] not in keep:
                print("removed stale", fp.relative_to(ROOT))
                fp.unlink()

# every (variant, mechanism, cumulative level) that has a CSV; a strategy that was
# never run is simply absent, so switching IMPUTER adds files instead of replacing them
rows, missing = [], 0
for variant in VARIANTS:
    for mech in MECHANISMS:
        for pct in PCT_LEVELS:
            stem = f"{SOURCE_SPLIT}_{DATASET}_long_{mech}_{pct}_{NULL_SEED}_{variant}"
            src = DATA_DIR / f"{stem}.csv"
            if not src.is_file():
                missing += 1
                continue
            out = to_ditto(src)
            assert out.name == stem.replace("_long", "") + ".txt", f"unexpected output name {out.name}"
            text = out.read_text(encoding="utf-8")
            assert all(l.count("\t") == 2 for l in text.splitlines()), f"{out.name}: not 3-field Ditto format"
            rows.append({"variant": variant, "imputer": variant.replace("imputed_", "") if "_" in variant else "-",
                         "mechanism": mech, "added %": STEPS[pct], "cumulative %": pct,
                         "file": out.name, "pairs": len(text.splitlines()),
                         "empty values": text.count("VAL \t") + text.count("VAL  ")})
print(f"wrote {len(rows)} files -> {DITTO_DIR.name}/   ({missing} variant/level combinations have no CSV)")

# the clean train / valid / test splits Ditto trains on, alongside the variants
for split in SPLITS:
    shutil.copy2(DATA_DIR / f"{split}_{DATASET}.txt", DITTO_DIR / f"{split}_{DATASET}.txt")

copied = [shutil.copy2(fp, SOLVER_DATA_DIR / fp.name) for fp in sorted(DITTO_DIR.glob("*.txt"))]
print(f"copied {len(copied)} files -> {SOLVER_DATA_DIR}")
pd.DataFrame(rows)

---
## Step 6 - combined evaluation summary

In [ ]:
# --- every parameter lives in the first cell; recovered from disk if this cell runs alone ---
if "PCT_LEVELS" not in globals():
    import os, sys
    from pathlib import Path
    _hint = os.environ.get("SOLVIER_ROOT")
    _cands = [Path(_hint).expanduser().resolve()] if _hint else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    _roots = [p for p in _cands if (p / "Data_preparation" / "Data").is_dir()]
    assert _roots, "run the first cell (it has PROJECT_ROOT), or export SOLVIER_ROOT=/path/to/SolvIER-main"
    _code = _roots[0] / "Data_preparation" / "code"
    assert (_code / "context.py").is_file(), f"context.py is missing - upload it to {_code}"
    sys.path.insert(0, str(_code))
    from context import load_context
    globals().update(load_context())
    print("settings recovered from", PARAMS_FP.name)

import sys

import pandas as pd

sys.path.insert(0, str(ROOT / "Data_preparation" / "code"))
from imputers import STRATEGIES  # noqa: E402

SUMMARY_FP = DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}_{SEED}_{IMPUTER}_imputation_summary.csv"
TUPLE_SUMMARY_FP = DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}_{SEED}_{IMPUTER}_tuple_similarity_summary.csv"
VALUE_SUMMARY_FP = DATA_DIR / f"{SOURCE_SPLIT}_{DATASET}_{SEED}_{IMPUTER}_value_similarity_summary.csv"

# read the files the parameters name, not whatever the folder happens to hold, so a
# leftover level or another seed cannot slip in. Strategies never run are skipped.
# The eval CSV already carries file/imputer/mechanism/null_pct/seed from
# save_evaluation, so we only enrich with the increment size ("added %") that is
# not in the file itself.
parts, tuple_parts, value_parts = [], [], []
for strategy in STRATEGIES:
    for mech in MECHANISMS:
        for pct in PCT_LEVELS:
            stem = f"{SOURCE_SPLIT}_{DATASET}_long_{mech}_{pct}_{NULL_SEED}_imputed_{strategy}"
            fp = DATA_DIR / f"{stem}_evaluation.csv"
            if fp.is_file():
                df = pd.read_csv(fp)
                df["added %"] = STEPS[pct]
                parts.append(df)
            # the per-tuple cosine CSV - stacked across every (mechanism x pct) with
            # the metadata columns the writer already prepended (file, imputer,
            # mechanism, null_pct, seed) so each row is self-describing
            tfp = DATA_DIR / f"{stem}_tuple_similarity.csv"
            if tfp.is_file():
                tuple_parts.append(pd.read_csv(tfp))
            # the per-cell cosine CSV - one row per imputed null, same metadata
            vfp = DATA_DIR / f"{stem}_value_similarity.csv"
            if vfp.is_file():
                value_parts.append(pd.read_csv(vfp))

assert parts, f"no *_imputed_*_evaluation.csv for {MECHANISMS} x {PCT_LEVELS} in {DATA_DIR}"
summary = pd.concat(parts, ignore_index=True)
summary = summary.sort_values(["imputer", "mechanism", "null_pct"]).reset_index(drop=True)
summary.to_csv(SUMMARY_FP, index=False)

print(f"combined {len(parts)} evaluation files from {sorted(summary.imputer.unique())} -> {SUMMARY_FP.name}")
print(f"total imputed cells across every file: {int(summary.total_imputed.sum())}")

if tuple_parts:
    tuple_summary = pd.concat(tuple_parts, ignore_index=True)
    tuple_summary = tuple_summary.sort_values(
        ["imputer", "mechanism", "null_pct", "row_index"]
    ).reset_index(drop=True)
    tuple_summary.to_csv(TUPLE_SUMMARY_FP, index=False)
    print(f"combined {len(tuple_parts)} tuple_similarity files -> {TUPLE_SUMMARY_FP.name}")
    print(f"total tuple rows across every file: {len(tuple_summary)}")
else:
    print("no *_tuple_similarity.csv found - skipped tuple summary")

if value_parts:
    value_summary = pd.concat(value_parts, ignore_index=True)
    value_summary = value_summary.sort_values(
        ["imputer", "mechanism", "null_pct", "row_index", "column"]
    ).reset_index(drop=True)
    value_summary.to_csv(VALUE_SUMMARY_FP, index=False)
    print(f"combined {len(value_parts)} value_similarity files -> {VALUE_SUMMARY_FP.name}")
    print(f"total value rows across every file: {len(value_summary)}")
else:
    print("no *_value_similarity.csv found - skipped value summary")

# the slim 14-column schema: see Data_preparation/code/imputers.EVAL_COLUMNS.
# * total_imputed             = K, the cells the imputer filled
# * imputed % (of all nulls)  = K / N * 100
# * semantic_similarity (... over imputed values) = cosine, averaged over K
# * semantic_similarity (... over all nulls)      = cosine, averaged over N
#                                                   (unfilled cells count 0)
# * semantic_similarity (... over imputed values that are not exact matches)
#                                                 = cosine, averaged over K - E
# * exact matches (over all nulls)      = E / N * 100
# * exact matches (over imputed values) = E / K * 100
summary